# Глава 26. Feature engineering: придумываем полезные признаки

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь создавать новые признаки из существующих (из дат, комбинаций, преобразований) и объяснить, почему хорошие признаки часто важнее выбора алгоритма.

На соревнованиях Kaggle есть поговорка: побеждает не тот, у кого сложнее модель, а тот, у кого лучше признаки. Feature engineering — искусство создавать из сырых данных признаки, которые помогают модели видеть закономерности. Это, возможно, самый недооценённый навык в ML.

**Аналогия: подсказки следователю  💡**

Модель — как следователь, а признаки — улики, которые ты ей даёшь. Можно свалить перед ней сырые данные («вот дата рождения: 2010-03-14»), а можно дать готовую подсказку («возраст: 15 лет, школьник»). Хороший инженер признаков превращает сырые данные в говорящие подсказки. Часто одна удачная подсказка помогает больше, чем замена следователя на более «умного».

## Приём 1: разбор дат

Дата «2026-03-14» сама по себе бесполезна для модели. Но из неё можно достать кучу полезного: день недели, месяц, сезон, выходной ли. Для прогноза продаж «выходной ли день» может решать всё.

In [ ]:
# === Подготовка: пингвины + синтетическая дата наблюдения ===
import pandas as pd, numpy as np
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url).dropna(subset=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "species"]).reset_index(drop=True)
# допустим, у каждого пингвина есть дата наблюдения:
df["date"] = pd.date_range("2023-01-01", periods=len(df), freq="D")

import pandas as pd

df["date"] = pd.to_datetime(df["date"])
df["день_недели"] = df["date"].dt.dayofweek   # 0=пн … 6=вс
df["месяц"] = df["date"].dt.month
df["выходной"] = df["день_недели"].isin([5, 6]).astype(int)  # сб/вс -> 1

## Приём 2: комбинирование признаков

Иногда сила не в отдельных признаках, а в их сочетании. На пингвинах отдельный признак не всегда так показателен, как сочетание: например, «лёгкий пингвин вида Adelie» выделяется чётче, чем масса или вид по отдельности.

In [ ]:
# Комбинируем массу и вид
df["лёгкий_адели"] = ((df["body_mass_g"] < 3500) & (df["species"] == "Adelie")).astype(int)

# Отношение двух измерений
df["клюв_к_ласту"] = df["bill_length_mm"] / df["flipper_length_mm"]

# Взаимодействие числовых признаков
df["клюв_x_глубина"] = df["bill_length_mm"] * df["bill_depth_mm"]

## Приём 3: преобразования чисел

Некоторые признаки «кривые»: например, цена билета, где большинство дешёвые, а редкие — огромные. Логарифмирование сглаживает такие перекосы. А биннинг (разбиение на группы) превращает число в категорию: возраст → «ребёнок / взрослый / пожилой».

In [ ]:
import numpy as np

# Логарифм сглаживает большие выбросы (log1p безопасен для нулей)
df["log_mass"] = np.log1p(df["body_mass_g"])

# Биннинг: масса -> весовые группы
df["класс_массы"] = pd.cut(df["body_mass_g"],
    bins=[0, 3500, 4500, 7000],
    labels=["лёгкий", "средний", "тяжёлый"])

## Локальный пример: цены на квартиры в Душанбе

Допустим, ты предсказываешь цену квартиры в Душанбе. Сырые признаки — площадь, этаж, район. Хорошие созданные признаки: «цена за квадратный метр района» (сильный сигнал), «первый или последний этаж» (обычно дешевле), «близко ли к центру», «новостройка ли» (из года постройки). Эти признаки-подсказки поднимут точность сильнее, чем смена модели.

## Мини-турнир: признаки против алгоритма

Проверим поговорку Kaggle на деле. Сравним: (а) сложная модель на сырых признаках и (б) простая модель на хороших признаках. Часто вариант (б) выигрывает.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# (а) Логистическая регрессия на сырых признаках
raw = df[["bill_length_mm", "bill_depth_mm", "body_mass_g"]].fillna(0)
print("Сырые:", cross_val_score(LogisticRegression(max_iter=1000),
                                 raw, df["species"], cv=5).mean().round(3))

# (б) Та же простая модель, но на созданных признаках
df["клюв_к_ласту"] = df["bill_length_mm"] / df["flipper_length_mm"]
feat = df[["bill_length_mm", "body_mass_g", "клюв_к_ласту"]].fillna(0)
feat["лёгкий_адели"] = ((df["body_mass_g"]<3500)&(df["species"]=="Adelie")).astype(int)
print("Признаки:", cross_val_score(LogisticRegression(max_iter=1000),
                                   feat, df["species"], cv=5).mean().round(3))

Часто вторая строка обгоняет первую — и это на той же простой модели, только с умными признаками. Вот почему feature engineering ценят так высоко.

## Задания

Базовый уровень

На пингвинах создай три новых признака: отношение длины клюва к длине ласта, «тяжёлый ли» (масса > 4500) и логарифм массы. Добавь их в модель и проверь по кросс-валидации — стало ли лучше?

Со звёздочкой  ⭐

Возьми датасет с датами (продажи, погода) и извлеки из даты минимум 4 признака (день недели, месяц, выходной, сезон). Проведи мини-турнир «сырые признаки vs созданные» на одной модели. Запиши прирост и объясни, какой признак помог больше всего.

Вопросы для обсуждения  💬

Почему одна удачная подсказка-признак может помочь модели больше, чем более сложный алгоритм?

Придумай три полезных признака для предсказания оценок ученика по данным о его учёбе.

Может ли созданный признак навредить модели? В каком случае?

**Проверь себя  ℹ️**

1. Что такое feature engineering и почему он так важен?

2. Какие признаки можно извлечь из даты?

3. Зачем логарифмировать признак с большими выбросами?